# Preparing Real Data

Simulated data arrives clean. Real data has missing values, odd codes, categories stored as text, and classes of very different sizes. This notebook walks through one real data set from start to finish. The story in one line: **before any model, look at the data; then put every step that learns from data inside one pipeline, so cross-validation stays honest.**

This notebook is a reference: read it on your own, and come back to it while you work on the Kaggle project and Lab 5. Unlike the lecture notebooks, almost all of its code is worth reading, because it's code you'll write yourself.

### By the end, you'll be able to

1. Look at a new data set and find its missing values, its odd values, and how balanced its classes are.
2. Choose between dropping and filling in (imputing) missing values, and explain why an imputer must be fit inside cross-validation.
3. Decide what to do about an outlier by first asking what the value means.
4. Turn categorical features into numbers with one-hot encoding, and explain why numbering the categories 0, 1, 2 is a problem for a linear model.
5. Build one pipeline that imputes, scales, encodes, and fits a model, and cross-validate it.
6. Explain why every step that learns from the data has to happen inside each fold.
7. Recognize class imbalance, compare accuracy with the majority-class floor, and respond with class weights or a different decision threshold.
8. Name the three families of feature-selection methods (filter, wrapper, embedded) and give an example of each.

In [ ]:
import numpy as np
import pandas as pd

**The data.** The Adult census data: about 49,000 adults from the 1994 US census, with the task of predicting whether a person's income is above \$50,000 a year. Like your Kaggle project, it's a classification task on a table of mixed numeric and text columns.

**Code worth reading.** `fetch_openml` downloads the data from OpenML the first time (you need an internet connection), then loads it from a saved copy afterward. `.frame` is the data as a pandas DataFrame.

We keep 10 of the columns. We leave out `race` and `sex`, to keep this notebook about data preparation (whether a model *should* use them is a real question, for another day); `fnlwgt`, a survey weight rather than a fact about the person; and `education`, which holds the same information as `education-num`.

In [ ]:
from sklearn.datasets import fetch_openml

adult = fetch_openml('adult', version=2, as_frame=True).frame

columns = ['age', 'workclass', 'education-num', 'marital-status', 'occupation',
           'relationship', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']
X = adult[columns]
y = (adult['class'] == '>50K').astype(int)      # 1 = income above $50K, 0 = not
X.head()

<a id="section-1"></a>

## 1 · Look first

Before any model, spend five minutes with the data. Four questions: What type is each column? What's missing? What looks odd? How big is each class?

**Code worth reading.** `.info()` lists each column's type and how many values are present. `.isna().sum()` counts the missing values in each column.

In [ ]:
X.info()

In [ ]:
X.isna().sum()

Three columns have missing values, and all three are categorical. (In the original file, missing values were written as `?`. OpenML already turned them into proper missing values, `NaN`. In a raw file, look for codes like `?`, `''`, `-1`, or `999` that stand for "missing.")

**Code worth reading.** `.describe()` summarizes the numeric columns. Look at each column's minimum and maximum: do they make sense?

In [ ]:
X.describe().round(1)

In [ ]:
y.value_counts(normalize=True).round(3)      # the share of each class

**Ask:** Two things in these outputs deserve a closer look. What are they?

<details>
<summary>Answer</summary>

First, `capital-gain` has a maximum of 99,999, far above everything else (its 75th percentile is 0). Section 3 looks at that. Second, the classes are unbalanced: only about 24% of people earn above \$50K. Section 7 looks at that.

</details>

<a id="section-2"></a>

## 2 · Missing values

There are two basic choices:

| Choice | When it's reasonable | The cost |
| --- | --- | --- |
| **Drop the rows** | Few rows are affected, or the *target* is missing (as with Hitters' salaries: we can't learn from a row with no answer) | Throws away data. And the test set will have missing values too, and you still have to predict for those rows. |
| **Fill them in (impute)** | Most other cases | The filled-in values are guesses. |

**Imputation** fills in a missing value with a guess: the column's median for a number, the most common category for a category, or a new category that says "missing." That last choice is often best for categories, because *being missing* can itself carry information.

**Code worth reading.** `SimpleImputer(strategy='constant', fill_value='missing')` replaces each missing value with the text `'missing'`.

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='constant', fill_value='missing')
filled = imputer.fit_transform(X[['occupation']])
print(f"'missing' now appears {(filled == 'missing').sum()} times; NaN appears {pd.isna(filled).sum()} times")

**An imputer learns from data.** `SimpleImputer(strategy='median')` has to compute the median, and it computes it from whatever data you `fit` it on. If you fit it on all the data before cross-validating, every fold's "training" median has peeked at that fold's validation rows. So the imputer has to be fit inside each fold, on that fold's training rows only. A pipeline does that for you (Section 5).

**Ask:** Is `occupation` missing at random, or could being missing tell us something? Check the income rate for the missing rows.

<details>
<summary>Answer</summary>

Run `y[X['occupation'].isna()].mean()`: about 9% of the people with a missing occupation earn above \$50K, against 24% overall. Being missing carries information (many of these people aren't working), which is why a separate `'missing'` category beats filling in the most common occupation.

</details>

In [ ]:
print(f"Income above $50K, occupation missing: {y[X['occupation'].isna()].mean():.1%}")
print(f"Income above $50K, everyone:           {y.mean():.1%}")

<a id="section-3"></a>

## 3 · Outliers: ask what the value means

An **outlier** is a value far from the rest. Plot first, then ask what the value *means* before you do anything. Most people have a capital gain of 0, so the plot shows only the nonzero values.

In [ ]:
from ch_6_and_8_demos_helpers import plot_capital_gain

plot_capital_gain(X['capital-gain']);

In [ ]:
top_coded = X['capital-gain'] == 99_999
print(f'{top_coded.sum()} people at exactly 99,999; the next largest value is '
      f"{X.loc[~top_coded, 'capital-gain'].max():,}")
print(f'Income above $50K among them: {y[top_coded].mean():.0%}')

Nobody's capital gain lands on exactly \$99,999 by chance, 244 times, with nothing between \$41,310 and \$99,999. It's a code: most likely the census recorded every gain at or above some cap as 99,999 (this is called **top-coding**). These are real people with very large gains, and every one of them earns above \$50K.

What to do, in order of preference:

- **Don't drop by default.** These rows are real, and the test data will have them too.
- **Know your model.** Trees barely care about extreme values: a split only asks which side of $s$ a value is on. Linear models can be pulled hard by them; standardizing doesn't fix that.
- **If it matters**, cap the column, or add a 0/1 column for "is top-coded." Compare the choices with cross-validation.

Here we leave the column as is.

**Ask:** A classmate deletes the 244 rows at 99,999 "because they're outliers." What goes wrong?

<details>
<summary>Answer</summary>

Those rows carry a strong signal (all of them earn above \$50K), so the model loses it. And the test data will still contain people at 99,999, which the model will now have never seen. Deleting is only right when a value is a genuine error, and even then, the test set may have the same error.

</details>

<a id="section-4"></a>

## 4 · Categorical features: one-hot encoding

Models need numbers. Why not number the categories, say `Private` = 0, `Self-emp-not-inc` = 1, `Local-gov` = 2? For a linear model, that says the categories are in order and equally spaced: that `Local-gov` is "twice" `Self-emp-not-inc`, and that one coefficient covers the whole list. Nothing about job types justifies that.

**One-hot encoding** makes one 0/1 column per category instead. Each row has a 1 in its own category's column and 0 everywhere else, so each category gets its own coefficient.

**Code worth reading.** `handle_unknown='ignore'` matters on real data: if a category shows up in new data that wasn't in the training data, its row gets all zeros instead of an error. `sparse_output=False` returns an ordinary table of numbers.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

small = pd.DataFrame({'workclass': ['Private', 'Local-gov', 'Private', 'State-gov']})
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit(small)
pd.DataFrame(encoder.transform(small), columns=encoder.get_feature_names_out()).astype(int)

In [ ]:
new_data = pd.DataFrame({'workclass': ['Federal-gov']})     # not seen during fit
encoder.transform(new_data)                                 # all zeros, no error

A two-valued category needs only one column (`drop='if_binary'`, as in the Hitters pipeline). A column with many categories, like `native-country` (41 countries), makes many columns; that's fine for most models, but worth knowing.

<a id="section-5"></a>

## 5 · One pipeline

Now put every step together. Numeric columns get imputed (with the median) and standardized; categorical columns get imputed (with `'missing'`) and one-hot encoded. Then a model.

**Code worth reading.**

- `make_pipeline(A, B)` chains steps: fitting the pipeline fits A, transforms the data with A, then fits B on the result.
- `make_column_transformer` sends each group of columns through its own steps and puts the results side by side.
- The numeric columns have no missing values in this data, but the imputer stays: new data might have some.

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

numeric_columns = ['age', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_columns = ['workclass', 'marital-status', 'occupation', 'relationship', 'native-country']

numeric_steps = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
categorical_steps = make_pipeline(SimpleImputer(strategy='constant', fill_value='missing'),
                                  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
preprocess = make_column_transformer((numeric_steps, numeric_columns),
                                     (categorical_steps, categorical_columns),
                                     verbose_feature_names_out=False)

logistic = make_pipeline(preprocess, LogisticRegression(max_iter=1000))

**Code worth reading.** Cross-validate the whole pipeline. In each of the 5 folds, `cross_val_score` fits the imputers, the scaler, the encoder, and the model on that fold's training rows only, then scores the validation rows. `DummyClassifier` always predicts the most common class: it's the **majority-class floor**, the accuracy you get without looking at any features. (`max_iter=1000` gives logistic regression enough steps to finish; its default `C=1` adds a ridge-style penalty, as in *Ridge, Lasso, and Ensembles*.)

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import KFold, cross_val_score

folds = KFold(n_splits=5, shuffle=True, random_state=0)
floor = cross_val_score(DummyClassifier(), X, y, cv=folds)
logistic_scores = cross_val_score(logistic, X, y, cv=folds)
print(f'Majority-class floor:  {floor.mean():.1%}')
print(f'Logistic regression:   {logistic_scores.mean():.1%} ± {logistic_scores.std():.1%}')

**Code worth reading.** A tree ensemble in the same kind of pipeline. Trees don't need scaling, so the numeric columns go straight through (`'passthrough'`). `HistGradientBoostingClassifier` is the fast boosting method from *Ridge, Lasso, and Ensembles*. Recent versions of scikit-learn let trees and tree ensembles handle missing values directly, without an imputer, but keeping one costs nothing and works with every model.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

tree_preprocess = make_column_transformer(('passthrough', numeric_columns),
                                          (categorical_steps, categorical_columns))
boosting = make_pipeline(tree_preprocess, HistGradientBoostingClassifier(random_state=0))
boosting_scores = cross_val_score(boosting, X, y, cv=folds)
print(f'Boosting:              {boosting_scores.mean():.1%} ± {boosting_scores.std():.1%}')

Both models use the **same folds** (`folds`), so the comparison is fair: every model is scored on exactly the same validation rows. Boosting is about 2 percentage points better, and the gap is several times the fold-to-fold spread.

<a id="section-6"></a>

## 6 · Why the pipeline matters: the wrong way to cross-validate

Here's a data set with nothing to learn: 50 samples with 5,000 features, every value random noise, and labels assigned by coin flip.

In [ ]:
rng = np.random.default_rng(465)
X_noise = rng.normal(size=(50, 5000))     # 50 samples, 5,000 features of pure noise
y_noise = rng.integers(0, 2, 50)          # coin-flip labels

**Ask:** These labels are coin flips. What error rate should an honest method get?

<details>
<summary>Answer</summary>

About 50%. There's nothing to learn, so any honest estimate should be close to guessing.

</details>

**Code worth reading: the wrong way.** `SelectKBest(f_classif, k=100)` keeps the 100 features most related to the labels. Here it looks at all 50 samples, and then we run 5-fold cross-validation of KNN with $K = 1$ on those 100 features.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.neighbors import KNeighborsClassifier

X_selected = SelectKBest(f_classif, k=100).fit_transform(X_noise, y_noise)    # looks at every label
scores = cross_val_score(KNeighborsClassifier(n_neighbors=1), X_selected, y_noise, cv=5)
print(f'Wrong way: cross-validation error = {1 - scores.mean():.0%}')

**Ask:** Same question for the right way, below: what error rate should it get?

<details>
<summary>Answer</summary>

About 50% again: the labels are still coin flips.

</details>

**Code worth reading: the right way.** The pipeline bundles the steps, so cross-validation repeats all of them in every fold. Feature selection now happens inside each fold, using only that fold's training points, on all 5,000 features.

In [ ]:
pipeline = make_pipeline(SelectKBest(f_classif, k=100), KNeighborsClassifier(n_neighbors=1))
scores = cross_val_score(pipeline, X_noise, y_noise, cv=5)
print(f'Right way: cross-validation error = {1 - scores.mean():.0%}')

In the wrong version, the feature selection looked at every label, including the labels of points that later served as validation points. The validation points weren't really new. The rule: **any step that learns from the data has to happen inside each fold.** That includes feature selection, imputers, scalers, and encoders. Putting them all in one pipeline is the easy way to obey it.

This mistake has appeared in published research, especially in genomics, where data sets have thousands of features and few samples. Imputers and scalers leak much less than feature selection does, but the habit is the same, and a pipeline costs nothing.

<a id="section-7"></a>

## 7 · Class imbalance

Only 24% of people earn above \$50K. That's **class imbalance**: one class is much rarer than the other. With imbalance, accuracy can mislead. The majority-class floor here is 76%, so "85% accuracy" is less impressive than it sounds. Chapter 4's metrics tell you more: **recall** (of the people above \$50K, what share did the model find?) and **precision** (of the people the model flagged as above \$50K, what share really are?). **F1** combines the two into one number.

**Code worth reading.** We need a set of rows to measure these on, so we hold out a validation set (stratified, so it has the same 24% as the whole). Then we compute each metric.

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, stratify=y,
                                                      random_state=0)
rows = []
for class_weight in [None, 'balanced']:
    model = make_pipeline(preprocess, LogisticRegression(max_iter=1000, class_weight=class_weight))
    predicted = model.fit(X_train, y_train).predict(X_valid)
    rows.append({'class_weight': str(class_weight),
                 'accuracy': np.mean(predicted == y_valid),
                 'recall': recall_score(y_valid, predicted),
                 'precision': precision_score(y_valid, predicted),
                 'F1': f1_score(y_valid, predicted)})
pd.DataFrame(rows).set_index('class_weight').round(3)

`class_weight='balanced'` tells the model to count each rare-class mistake more heavily, in proportion to how rare the class is. Recall jumps (from about 60% to 84%), precision falls, and accuracy drops. Whether that's better depends on what each kind of mistake costs, which is a question about the problem, not the model.

**Moving the threshold** is the other tool, straight from Chapter 4. The model predicts "above \$50K" when its estimated probability is at least 0.5, but nothing forces 0.5.

**Code worth reading.** `predict_proba(...)[:, 1]` is each person's estimated probability of class 1.

In [ ]:
from ch_6_and_8_demos_helpers import plot_threshold_tradeoff

model = make_pipeline(preprocess, LogisticRegression(max_iter=1000)).fit(X_train, y_train)
probability = model.predict_proba(X_valid)[:, 1]
plot_threshold_tradeoff(y_valid, probability, marked=(0.3, 0.5));

In [ ]:
predicted = probability >= 0.3
print(f'Threshold 0.3: recall {recall_score(y_valid, predicted):.1%}, '
      f'precision {precision_score(y_valid, predicted):.1%}')

Lowering the threshold to 0.3 flags more people: recall rises, precision falls. Choose the threshold on validation data (or by cross-validation), never on the test set. A third tool, **resampling** (copying rare-class rows, or dropping common-class rows, before fitting), exists too; class weights usually do the same job more simply.

And always report the majority-class floor next to your accuracy. Your Kaggle checkpoint asks for it for this reason.

**Ask:** A model for a rare disease (1% of patients) has 99% accuracy. Is it any good?

<details>
<summary>Answer</summary>

You can't tell from accuracy. Predicting "healthy" for everyone also gets 99%, with a recall of 0%: it finds no sick patients at all. Compare against the majority-class floor and look at recall and precision.

</details>

<a id="section-8"></a>

## 8 · Feature selection: three families

Sometimes you want fewer features: a simpler model, faster training, or less variance. The methods come in three families:

| Family | How it works | Examples |
| --- | --- | --- |
| **Filter** | Score each feature on its own against the target, keep the top ones, then fit the model | `SelectKBest` (Section 6); ranking features by their correlation with the target |
| **Wrapper** | Try different subsets of features, fit the model on each, and compare their CV scores | Best subset and forward/backward stepwise selection (ISL §6.1, not covered in this course) |
| **Embedded** | The model chooses features as part of fitting | The lasso's exact zeros; a tree's feature importances |

Every one of them looks at the labels, so every one belongs inside the pipeline (Section 6).

**Code worth reading.** A tree's `feature_importances_` says how much each feature's splits lowered the Gini impurity, as a share of the total. Features the tree never split on get 0. Here's a tree with `max_depth=6` (a limit, from *Trees and Regularization*) in the same pipeline.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from ch_6_and_8_demos_helpers import plot_importances

tree = make_pipeline(preprocess, DecisionTreeClassifier(max_depth=6, random_state=0)).fit(X, y)
names = tree[0].get_feature_names_out()      # the columns after one-hot encoding
importances = tree[-1].feature_importances_
print(f'{np.sum(importances == 0)} of {len(importances)} columns have importance 0')
plot_importances(importances, names);

The tree leans on being married, education, and capital gains and losses, and most of the 84 columns (71 of them) get no splits at all. One warning: impurity importances tend to favor numeric features with many distinct values, so treat them as a rough guide, not a verdict.

<a id="section-9"></a>

## 9 · Summary: a checklist for a new data set

1. **Look first:** `.info()`, `.isna().sum()`, `.describe()`, and the class shares. Look for codes that mean "missing" and values that can't be right.
2. **Missing values:** drop rows only when the target is missing (or very few rows are affected); otherwise impute. A `'missing'` category is often best for categorical columns.
3. **Outliers:** plot, then ask what the value means. Don't drop by default. Trees barely care; linear models can.
4. **Categories:** one-hot encode, with `handle_unknown='ignore'`.
5. **One pipeline:** imputers, scaler (not needed for trees), encoder, any feature selection, and the model, all inside one pipeline, cross-validated with the same folds for every model.
6. **Imbalance:** report the majority-class floor; look at recall, precision, and F1; try `class_weight='balanced'` or a different threshold, chosen on validation data.

# Practice Questions

These questions cover missing values, outliers, encoding, pipelines, class imbalance, and feature-selection families. Questions like these will appear on the learning check, so try each one on paper before you open its hint or answer.

### Question 1: Drop or impute?

For each situation, would you drop the rows or impute? (See [Section 2](#section-2).)

(a) 59 rows have no value for the target.

(b) 6% of rows have no value for one categorical feature, and the test set has missing values in the same column.

<details>
<summary>Answer</summary>

(a) **Drop.** A row with no target has nothing to learn from.

(b) **Impute**, for example with a `'missing'` category. Dropping throws away 6% of the data, and you'd still have to predict for the test rows with missing values.

</details>

### Question 2: Where to fit the imputer

A classmate fits `SimpleImputer(strategy='median')` on the whole training set, then runs 5-fold cross-validation on the imputed data. What's wrong, and how do you fix it? (See [Section 2](#section-2) and [Section 6](#section-6).)

<details>
<summary>Answer</summary>

**The medians were computed using every fold's validation rows**, so the validation rows weren't truly new. Put the imputer in a pipeline with the model, and cross-validate the pipeline, so the median is recomputed from each fold's training rows only.

</details>

### Question 3: An odd value

244 people have a capital gain of exactly 99,999, and the next largest value is 41,310. What's the most likely explanation, and should you delete those rows? (See [Section 3](#section-3).)

<details>
<summary>Answer</summary>

**It's a code (top-coding): gains at or above a cap were recorded as 99,999.** Don't delete the rows: they're real people, they carry a strong signal (all earn above \$50K), and the test data will contain them too.

</details>

### Question 4: Numbering categories

A column `color` has values red, green, and blue. A classmate codes them 0, 1, 2 and fits a linear regression.

(a) What does that coding assume?

(b) How many 0/1 columns does one-hot encoding make for this column (without dropping any)? (See [Section 4](#section-4).)

<details>
<summary>Answer</summary>

(a) **That the colors are in order and equally spaced**, so one coefficient covers them all and blue's effect is exactly twice green's.

(b) **3**, one per color.

</details>

### Question 5: The wrong way

On coin-flip labels with 5,000 noise features, choosing the 100 best features on all the data and then cross-validating gave a 2% error rate. Doing the selection inside each fold gave 46%. Which number is the honest estimate, and what's the general rule? (See [Section 6](#section-6).)

<details>
<summary>Answer</summary>

**46%** (near the 50% you'd expect from coin flips). **Any step that learns from the data, especially from the labels, has to happen inside each fold**, which a pipeline does automatically.

</details>

### Question 6: Imbalance

In a data set, 24% of rows are class 1.

(a) What accuracy does the majority-class floor get?

(b) Setting `class_weight='balanced'` raises recall from 60% to 84%. What usually happens to precision? (See [Section 7](#section-7).)

(c) Name one other way to raise recall without refitting with class weights.

<details>
<summary>Answer</summary>

(a) **76%**, by predicting class 0 for everyone.

(b) **It usually falls**, because the model flags more rows as class 1, including more that aren't.

(c) **Lower the decision threshold** on the predicted probabilities (for example from 0.5 to 0.3), chosen on validation data.

</details>

### Question 7: Feature-selection families

Label each as a **filter**, **wrapper**, or **embedded** method. (See [Section 8](#section-8).)

(a) The lasso sets 6 of 19 coefficients to exactly zero.

(b) Keep the 10 features with the highest correlation with the target, then fit a model.

(c) Fit the model on many different subsets of features and keep the subset with the best CV score.

(d) A tree never splits on 80 of its 100 columns.

<details>
<summary>Answer</summary>

(a) **Embedded.** (b) **Filter.** (c) **Wrapper.** (d) **Embedded** (the tree chose features as part of fitting).

</details>